# **Section 5:** Modern Geospatial Analytics: Python + SpatialSQL

This is the finale of the workshop, and it's meant to feel different from the first three hours. We're not going to teach every feature of DuckDB — the goal is to show you a **new way of thinking** about geospatial analysis.

```
CSV / Parquet
      ↓
    DuckDB
      ↓
     SQL
      ↓
 DataFrame
      ↓
    Python
```


## Initial Setup

The setup here is simple:

1. Install `duckdb`
2. Import: `os`, `urllib.requests`, `duckdb`
3. Download `311_requests.csv` from GitHub
4. Load DuckDB extentions: `spatial`, `httpfs`  

This code is safe to re-run any time, including right after a reconnect.

In [ ]:
# Let's first install duckdb
%pip install -qq duckdb

In [ ]:
# Checking if 311_requests.csv is still available from Notebook 1
# If not, it redownloads the file from GitHub
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/nmholman/ubiquitous-barnacle/main"  # same repo as Notebook 01

import os, urllib.request
if not os.path.exists("311_requests.csv"):
    urllib.request.urlretrieve(f"{GITHUB_RAW_BASE}/data/311_requests.csv", "311_requests.csv")


print("Ready: 311_requests.csv present")


## 5.1 DuckDB Fundamentals

[DuckDB](https://duckdb.org/) is an **in-process analytical database** designed for fast querying and analysis of tabular data. Unlike traditional databases such as PostgreSQL, DuckDB does not require a separate database server—it runs directly within your Python process.

DuckDB can:

- Query CSV, Parquet, and other data files directly
- Perform fast SQL queries and aggregations
- Work alongside pandas, [GeoPandas](https://geopandas.org/en/stable/), and NumPy
- Use extensions to add capabilities such as spatial analysis

### Creating a DuckDB Database

Getting started is simple. Install the Python package:

```
pip install duckdb
```

Then create or connect to a database with just a few lines:

```
import duckdb
con = duckdb.connect("workshop.db")
```

If the database file doesn't exist, DuckDB creates it automatically. You can also create an **in-memory database** without creating a file:

```
con = duckdb.connect()
```

You can then execute SQL directly from Python:

```
con.sql("""
    SELECT *
    FROM '311_requests.csv'
    LIMIT 5
""")
```

This ability to **query files directly without first loading them into a database** is one of DuckDB's most useful features for data analysis.

### DuckDB Spatial

DuckDB's **Spatial extension** adds geospatial data types and functions, including familiar GIS operations such as area, distance, buffers, intersections, and spatial relationships.

```
INSTALL spatial;
LOAD spatial;
```

Once loaded, you can use spatial functions such as:

```
SELECT
    name,
    ST_Area(geometry) AS area
FROM counties;
```

### DuckDB Documentation

- [DuckDB Documentation](https://duckdb.org/docs/current/)
- [Spatial Data Management with DuckDB](https://duckdb.gishub.org/)
- [DuckDB Spatial Extension](https://duckdb.org/docs/current/core_extensions/spatial/overview.html)
- [Spatial Function Reference](https://duckdb.org/docs/current/core_extensions/spatial/functions.html)
- [DuckDB SQL Intro](https://duckdb.org/docs/current/sql/introduction)

### Let's get into it!

**Your turn!** We have already installed `duckdb` above, so now let's import the library and create a connection in the codeblock below:

In [ ]:
# Import DuckDB on the line below:



# Create & connect to a database



# Install and load extensions
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("httpfs")
con.load_extension("httpfs")

# Testing the connection
con.sql("SELECT current_database()")

<details>
<summary>Answers</summary>

```
# Import DuckDB
import duckdb

# Create & connect to an in-memory database
con = duckdb.connect("workshop.db")

# Install & load extensions
con.install_extension("spatial")
con.load_extension("spatial")
con.install_extension("httpfs")
con.load_extension("httpfs")
```

</details>

Great! Now let's read our `311_requests.csv` data using SQL:

In [ ]:
con.sql("""
    SELECT request_type, COUNT(*) AS requests
    FROM '311_requests.csv'
    GROUP BY request_type
    ORDER BY requests DESC
""")


That query didn't load the CSV into a Python variable or create a table in the database. **DuckDB read the file directly and queried it in place.**

If you want to continue working with the results in Pandas, you can easily pull the query result into a DataFrame:


In [ ]:
# Loading the query results into Pandas DataFrame
result_df = con.sql("""
    SELECT department, COUNT(*) AS requests
    FROM '311_requests.csv'
    GROUP BY department
    ORDER BY requests DESC
""").df()

result_df


Alternatively, you can load the data into a DuckDB table and query it there:

```
con.sql("""
    CREATE TABLE requests AS
    SELECT *
    FROM '311_requests.csv'
""")
```

Now the data is stored in the DuckDB database, so you can query the table directly:

```
con.sql("""
    SELECT department, COUNT(*) AS requests
    FROM requests
    GROUP BY department
    ORDER BY requests DESC
""")
```

This gives you two approaches:

<br>

| Query the file directly | Load into DuckDB |
|---|---|
| No need to create a table | Data is stored in the database |
| Great for one-off analysis | Useful for repeated analysis |
| Changes to the source file are automatically reflected | Queries don't need to re-read the source file |
| Less setup | Can be faster for repeated queries |

<br>

**Rule of thumb:** If you're exploring a file or only need to query it once or twice, querying it directly is convenient. If you're going to work with the data repeatedly, loading it into DuckDB can make more sense.

<br>


---




## 5.2 Overture Maps

So far, we've worked with datasets that fit comfortably on our computers. What happens when we start working with **much larger geospatial datasets**?

**Overture Maps** is an open, large-scale geospatial dataset covering themes including:

- **Places** — points of interest, businesses, amenities, etc.
- **Buildings** — building footprints
- **Transportation** — roads, paths, and connectors
- **Divisions** — administrative boundaries
- **Base** — land, land use, water, and other foundational features

Overture contains **billions of features**, so downloading an entire theme just to answer a simple question isn't practical.

The important shift in thinking is:

> Instead of treating a massive geospatial dataset as something we have to download and load into a desktop GIS, we can approach it as something we want to **query**.

Overture publishes its data as **GeoParquet** files in public cloud storage. DuckDB can query these files directly from the cloud, allowing us to select only the columns and geographic areas we need rather than downloading the entire dataset.

### A quick note about Parquet

**[Parquet](https://parquet.apache.org/docs/)** is a column-oriented file format designed for efficient data analysis. Unlike formats such as CSV, which store data row by row, Parquet organizes data by column. This makes it particularly efficient when we only need a few columns from a large dataset.

[**GeoParquet**](https://geoparquet.org/) extends Parquet with metadata for storing geospatial data, including geometry and coordinate reference system information. It has become an important format for working with large, cloud-hosted geospatial datasets.

This makes Parquet a great fit for the kind of workflow we're about to explore:

**Cloud storage → DuckDB → targeted query → only the data we need**

### Overture Documentation

- [Overture Maps documentation](https://docs.overturemaps.org/) | Detailed information about the datasets, schemas, and ways to access the data.
- [Overture Quickstart](https://docs.overturemaps.org/getting-data/) | Overview of the different ways to access Overture data
- [Querying Overture with DuckDB](https://docs.overturemaps.org/getting-data/duckdb/) | Examples of querying Overture's cloud-hosted GeoParquet with DuckDB
- [Accessing the Overture Catalog](https://docs.overturemaps.org/getting-data/cloud-sources/) | Information about Overture's cloud storage and data organization
- [Overture Schema Reference](https://docs.overturemaps.org/schema/reference/) | Detailed information about the fields available in each dataset

In [ ]:
# Overture Maps release used for this workshop.
# Check https://docs.overturemaps.org/release-calendar/ if you want the newest release string.
OVERTURE_RELEASE = "2026-08-19.0"

BUILDINGS_PATH = f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=buildings/type=building/*"
TRANSPORTATION_PATH = f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=transportation/type=segment/*"
PLACES_PATH = f"s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=places/type=place/*"

con.sql("SET s3_region='us-west-2';")

## 5.3 A Geospatial Analysis at Scale

**GIVE ME COFFEE, PLEASE:** This section combines three things at once, which is a very realistic shape for a real GIS project: a private open dataset (Overture Places), a government dataset (Georgia county boundaries from a live ArcGIS REST service), and a bit of math (population, for a fair per-capita comparison).

```
Overture (coffee shops)      ArcGIS REST (county boundaries + population)
          |                              |
          `--------------+---------------'
                          |
            Spatial join (point-in-polygon)
                          |
               Coffee shops per county
                          |
                     Coffee Map
                          |
             Coffee shops near MARTA stations
```

<br>

### **Step 1:** Coffee shops in Georgia (Overture)

We already defined `PLACES_PATH` back in 4.2. Overture's places schema tags every point of interest with a `categories.primary` value — for coffee shops, that value is, you guessed it, `'coffee_shop'`.

In [ ]:
import pandas as pd

# Georgia's approximate statewide bounding box
GA_BBOX = (-85.61, 30.36, -80.84, 35.00)
ga_min_lon, ga_min_lat, ga_max_lon, ga_max_lat = GA_BBOX

ga_coffee_shops = con.sql(f"""
    SELECT
        names.primary AS name,
        brand.names.primary AS brand,
        geometry AS geom
    FROM read_parquet('{PLACES_PATH}', filename=true, hive_partitioning=1)
    WHERE categories.primary = 'coffee_shop'
      AND bbox.xmin BETWEEN {ga_min_lon} AND {ga_max_lon}
      AND bbox.ymin BETWEEN {ga_min_lat} AND {ga_max_lat}
""")

con.sql("CREATE OR REPLACE TABLE ga_coffee_shops AS SELECT * FROM ga_coffee_shops")
con.sql("SELECT COUNT(*) AS total_coffee_shops FROM ga_coffee_shops")


**Let's explore the data a bit.** Most common named brands, with everything else grouped as independent. Any guesses?

In [ ]:
con.sql("""
    SELECT
        COALESCE(brand, 'Independent / unbranded') AS brand,
        COUNT(*) AS locations
    FROM ga_coffee_shops
    GROUP BY brand
    ORDER BY locations DESC
    LIMIT 15
""")


### **Step 2:** Georgia county boundaries (ArcGIS REST service)

This is a public ArcGIS REST endpoint from SAGIS portal. We're pulling it as GeoJSON and letting DuckDB's spatial extension read it directly.

In [ ]:
import urllib.request

COUNTIES_URL = "https://pub.sagis.org/arcgis/rest/services/OpenData/Boundaries/MapServer/15/query?outFields=*&where=1%3D1&f=geojson"
urllib.request.urlretrieve(COUNTIES_URL, "ga_counties.geojson")

con.sql("CREATE OR REPLACE TABLE ga_counties AS SELECT * FROM ST_Read('ga_counties.geojson')")
con.sql("SELECT COUNT(*) AS county_count FROM ga_counties")


**Check your schema!** here's what we got:

In [ ]:
con.sql("SELECT NAME10, GEOID10, totpop10 FROM ga_counties LIMIT 5")


We'll also need the raw GeoJSON as a plain Python dictionary later, for Plotly's choropleth since it needs the actual GeoJSON structure, not a database table.

In [ ]:
import json

with open("ga_counties.geojson") as f:
    ga_counties_geojson = json.load(f)

print(f"{len(ga_counties_geojson['features'])} county features loaded")


### **Step 3:** Coffee shops per county

A point-in-polygon spatial join: for every coffee shop, which county polygon contains it? `LEFT JOIN` keeps every county even if it has zero coffee shops.

In [ ]:
coffee_by_county = con.sql("""
    SELECT
        c.GEOID10 AS geoid,
        c.NAME10 AS county,
        c.totpop10 AS population,
        COUNT(s.name) AS coffee_shop_count
    FROM ga_counties c
    LEFT JOIN ga_coffee_shops s
      ON ST_Within(s.geom, c.geom)
    GROUP BY c.GEOID10, c.NAME10, c.totpop10
""").df()

# Per-capita metric, since a big county will always have more shops than a small one --
# guard against any county with population 0 or missing.
coffee_by_county["coffee_shops_per_10k"] = (
    coffee_by_county["coffee_shop_count"] / coffee_by_county["population"].replace(0, pd.NA) * 10000
)

coffee_by_county.sort_values("coffee_shop_count", ascending=False).head(10)


**Switching data sources for the map geometry.** The SAGIS ArcGIS REST export we used for county names and population is fine for that — but its GeoJSON geometry renders incorrectly in Plotly regardless of winding-order fixes, which points to a deeper topology issue in that specific export rather than something fixable with a quick correction.

Rather than keep patching it, let's swap to a source that's proven to work with Plotly: the [Census-derived county GeoJSON](https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json) that Plotly's own documentation uses in nearly every US county choropleth example. We only need it for the *shapes* — population and the coffee-shop counts we already calculated in Step 3 came from SAGIS and don't need to change.

In [ ]:
import urllib.request

with urllib.request.urlopen(
    "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
) as response:
    us_counties_geojson = json.load(response)

# Georgia's state FIPS prefix is "13" -- keep only Georgia's counties.
# Each feature's top-level "id" is already the 5-digit county FIPS code,
# same format as the "geoid" column in coffee_by_county.
ga_counties_geojson = {
    "type": "FeatureCollection",
    "features": [f for f in us_counties_geojson["features"] if f["id"].startswith("13")],
}

print(f"{len(ga_counties_geojson['features'])} Georgia county features loaded (expect 159)")


### **Step 4:** Choropleth map (Plotly Express)

Using `plotly.express` for ease. Just give it a GeoJSON, a DataFrame, and a column to color by, and it handles the map projection and legend for you.

In [ ]:
import plotly.express as px

# Guard against a dtype mismatch between our geoid column and the GeoJSON key
# -- if these don't match exactly, Plotly silently colors nothing.
coffee_by_county["geoid"] = coffee_by_county["geoid"].astype(str)

# Guard against a single tiny-population county blowing out the color scale --
# clip the color range at the 95th percentile so one outlier doesn't wash out
# every other county's color.
color_cap = coffee_by_county["coffee_shops_per_10k"].quantile(0.95)

fig = px.choropleth(
    coffee_by_county,
    geojson=ga_counties_geojson,
    locations="geoid",
    featureidkey="id",  # new source keys features by FIPS at the top level, not nested under properties
    color="coffee_shops_per_10k",
    range_color=(0, color_cap),
    color_continuous_scale="YlOrBr",
    hover_name="county",
    hover_data={"coffee_shop_count": True, "population": True, "geoid": False},
    title="Coffee Shops per 10,000 Residents by Georgia County",
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    height=700,
    coloraxis_colorbar_title="Per 10k residents",
)
fig.show()

## 5.4 Compare the Workflows

**Traditional desktop workflow:**
```
Acquire data → Prepare data → Load into desktop GIS → Project →
Clip/filter → Spatial analysis → Summarize → Export
```

**Python + DuckDB workflow:**
```
Query → Analyze → Summarize → Visualize
```
<br>

>This isn't about replacing desktop GIS. It's about adding tools to your toolbox!

## Final Wrap-Up

**Three takeaways:**

1. **Python doesn't have to be intimidating.** You can start by reading and modifying existing scripts.
2. **The ArcGIS API can automate repetitive GIS administration.** You're leaving with `content_inventory.py`, `content_audit.py`, and `inactive_users.py`, ready to adapt for your organization.
3. **GIS doesn't have to happen entirely inside desktop GIS.** Python + DuckDB + modern open geospatial datasets open up analytical workflows that are difficult or cumbersome to perform interactively.

> You don't have to choose between ArcGIS and open-source tools. The real skill is knowing which tool is appropriate for the problem you're trying to solve.